# RegDrift: run the pipeline
Runs on Colab's free CPU. Add secrets (key icon, left): `GROQ_API_KEY`, and for Jira `JIRA_URL`, `JIRA_EMAIL`, `JIRA_TOKEN`, `JIRA_PROJECT`.
Nothing is created in Jira unless all four Jira secrets are set and you do **not** pass `--dry-run`.

In [ ]:
REPO_URL = "https://github.com/Ajitesh1234/RegDrift.git"  # change if your repo name differs
!git clone $REPO_URL repo
%cd repo
!pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import userdata
for k in ["GROQ_API_KEY", "JIRA_URL", "JIRA_EMAIL", "JIRA_TOKEN", "JIRA_PROJECT"]:
    try: os.environ[k] = userdata.get(k)
    except Exception: print("not set (optional):", k)

## Analyse one circular (safe: no Jira tickets)

In [ ]:
!python regdrift.py circulars/rbi_fx_derivatives_2026-10-10.pdf --dry-run

## Analyse every circular in the folder and create (de-duplicated) Jira tickets
Remove `--dry-run` only when you want real tickets.

In [ ]:
!python regdrift.py circulars/ --dry-run

## Audit report
Every run writes `reports/<circular>.report.json`: input hash, model used, all obligations, every finding (including compliant ones), and ticket keys.

In [ ]:
import json, glob
for p in sorted(glob.glob("reports/*.json")):
    r = json.load(open(p))
    print(p, "|", r["circular_ref"], "|", r["extractor"], r["llm_model"], "|", r["counts"])

## Review tickets (feeds the LoRA training data)
`%run` (not `!python`) so the prompts accept input in Colab.

In [ ]:
%run review.py